# Loan Default Prediction — Modelling

**Objective:** Train and evaluate classification models to predict loan default, optimising for recall on the default class.

**Business context:** A missed default (false negative) is far more costly than a false alarm. We tune classification thresholds to maximise recall while maintaining acceptable precision.

**Models:**
1. Logistic Regression — interpretable baseline
2. XGBoost — primary model
3. LightGBM — challenger model
4. Threshold optimisation on best model
5. SHAP feature importance

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../data/application_train_processed.csv')
print(f'Shape: {df.shape}')

X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']

print(f'Features: {X.shape[1]}')
print(f'Class distribution: {y.value_counts(normalize=True).round(3).to_dict()}')

## 2. Train / Validation Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} rows | Default rate: {y_train.mean():.3f}')
print(f'Val:   {X_val.shape[0]:,} rows | Default rate: {y_val.mean():.3f}')

## 3. Evaluation Helper

In [ ]:
def evaluate_model(name, model, X_val, y_val, threshold=0.5):
    proba = model.predict_proba(X_val)[:, 1]
    preds = (proba >= threshold).astype(int)

    roc_auc = roc_auc_score(y_val, proba)
    avg_prec = average_precision_score(y_val, proba)

    print(f'\n=== {name} (threshold={threshold}) ===')
    print(f'ROC-AUC:          {roc_auc:.4f}')
    print(f'Avg Precision:    {avg_prec:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_val, preds, target_names=['No Default', 'Default']))

    return proba, roc_auc, avg_prec

## 4. Baseline — Logistic Regression

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=RANDOM_STATE
    ))
])

lr_pipeline.fit(X_train, y_train)
lr_proba, lr_auc, lr_ap = evaluate_model('Logistic Regression', lr_pipeline, X_val, y_val)

## 5. XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.1f}')

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    verbosity=0
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f'Best iteration: {xgb.best_iteration}')
xgb_proba, xgb_auc, xgb_ap = evaluate_model('XGBoost', xgb, X_val, y_val)

## 6. LightGBM

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    verbose=-1
)

lgbm.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[]
)

lgbm_proba, lgbm_auc, lgbm_ap = evaluate_model('LightGBM', lgbm, X_val, y_val)

## 7. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = [
    ('Logistic Regression', lr_proba, lr_auc, '#9E9E9E'),
    ('XGBoost',             xgb_proba, xgb_auc, '#1565C0'),
    ('LightGBM',            lgbm_proba, lgbm_auc, '#2E7D32'),
]

# ROC Curve
for name, proba, auc, color in models:
    fpr, tpr, _ = roc_curve(y_val, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend()

# Precision-Recall Curve
for name, proba, auc, color in models:
    prec, rec, _ = precision_recall_curve(y_val, proba)
    ap = average_precision_score(y_val, proba)
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})', color=color)
axes[1].axhline(y=y_val.mean(), color='k', linestyle='--', alpha=0.4, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.suptitle('Model Comparison', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Threshold Optimisation

The default 0.5 threshold is not optimal for imbalanced credit risk problems. We find the threshold that maximises F2 score (which weights recall more heavily than precision).

In [ ]:
# Use best model (XGBoost or LightGBM based on AUC)
best_proba = xgb_proba if xgb_auc >= lgbm_auc else lgbm_proba
best_name = 'XGBoost' if xgb_auc >= lgbm_auc else 'LightGBM'
print(f'Best model: {best_name}')

thresholds = np.arange(0.1, 0.9, 0.01)
results = []

for t in thresholds:
    preds = (best_proba >= t).astype(int)
    tp = ((preds == 1) & (y_val == 1)).sum()
    fp = ((preds == 1) & (y_val == 0)).sum()
    fn = ((preds == 0) & (y_val == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f2        = (5 * precision * recall) / (4 * precision + recall) if (precision + recall) > 0 else 0

    results.append({'threshold': t, 'precision': precision, 'recall': recall, 'f2': f2})

results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['f2'].idxmax()]
print(f'\nOptimal threshold (max F2): {best_row["threshold"]:.2f}')
print(f'  Precision: {best_row["precision"]:.3f}')
print(f'  Recall:    {best_row["recall"]:.3f}')
print(f'  F2:        {best_row["f2"]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(results_df['threshold'], results_df['precision'], label='Precision', color='#1565C0')
ax.plot(results_df['threshold'], results_df['recall'],    label='Recall',    color='#C62828')
ax.plot(results_df['threshold'], results_df['f2'],        label='F2 Score',  color='#2E7D32', linewidth=2)
ax.axvline(x=best_row['threshold'], color='black', linestyle='--', alpha=0.6,
           label=f'Optimal threshold = {best_row["threshold"]:.2f}')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title(f'{best_name} — Threshold Optimisation (F2)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate best model at optimal threshold
best_model = xgb if xgb_auc >= lgbm_auc else lgbm
evaluate_model(f'{best_name} (optimised)', best_model, X_val, y_val,
               threshold=best_row['threshold'])

## 9. Confusion Matrix

In [ ]:
opt_preds = (best_proba >= best_row['threshold']).astype(int)
cm = confusion_matrix(y_val, opt_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: No Default', 'Pred: Default'],
            yticklabels=['True: No Default', 'True: Default'], ax=ax)
ax.set_title(f'{best_name} — Confusion Matrix (threshold={best_row["threshold"]:.2f})', fontweight='bold')
plt.tight_layout()
plt.show()

## 10. SHAP Feature Importance

In [ ]:
# Use a sample for SHAP (full dataset is slow)
X_sample = X_val.sample(2000, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, max_display=20, show=False)
plt.title('SHAP Feature Importance — Top 20', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bar plot of mean absolute SHAP values
shap_importance = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(shap_importance['feature'][::-1], shap_importance['mean_abs_shap'][::-1], color='#1565C0')
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 20 Features by Mean Absolute SHAP Value', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    {'Model': 'Logistic Regression', 'ROC-AUC': lr_auc,   'Avg Precision': lr_ap,   'Threshold': 0.5},
    {'Model': 'XGBoost',             'ROC-AUC': xgb_auc,  'Avg Precision': xgb_ap,  'Threshold': 0.5},
    {'Model': 'LightGBM',            'ROC-AUC': lgbm_auc, 'Avg Precision': lgbm_ap, 'Threshold': 0.5},
    {'Model': f'{best_name} (tuned)', 'ROC-AUC': xgb_auc if best_name=='XGBoost' else lgbm_auc,
     'Avg Precision': xgb_ap if best_name=='XGBoost' else lgbm_ap,
     'Threshold': round(best_row['threshold'], 2)},
])

summary

**Next:** Business recommendations in `04_recommendations.ipynb`